## LIBRARIES

In [2]:
import pandas as pd
import numpy as np
import time
import warnings
import base64
from io import BytesIO
import os
import ipywidgets as widgets
from IPython.display import display, HTML

# Scikit-learn components
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest
from sklearn.tree import plot_tree

# Plotly components
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Matplotlib component
import matplotlib.pyplot as plt

import warnings
# Ignora tutti i FutureWarning di pandas
warnings.simplefilter(action='ignore', category=FutureWarning)


## PREPARATION

### Vessel Data Acquisition and Feature Selection


In [3]:
# raw data source paths
DataFrames = {
    "ECO ADRIATICA": r"C:\Users\siani\Desktop\ECO ADRIATICA_till.csv"}
file_path2 = r"C:\Users\siani\Desktop\port_id-Battery 1.csv"

# feature group classification
navigation_vars = ['ep_SHIP_SOG_1', 'ep_SHIP_STW_1', 'ep_SHIP_DRAFTAFT_1', 'ep_SHIP_DRAFTFOR_1', 'ep_SHIP_HEAD_1', 'miglia']
power_vars      = ['ep_SHA_POW_1', 'ep_SHA_POW_2', 
                   'ep_SHG_POW_1', 'ep_SHG_POW_2']
weather_vars    = ['ep_SHIP_WSPEED_1', 'ep_WH_WAVEH_1', 'ep_WH_WAVEP_1', 'ep_WH_SWELLH_1', 'ep_WH_SWELLP_1',
                   'ep_WH_SPEED_1', 'ep_SHIP_WDIR_1', 'ep_SHIP_SEAF_1', 'ep_SHIP_SEADIR_1', 'ep_WH_DIR_1', 'ep_WH_AIRT_1']
position_vars   = ['ep_SHIP_LAT_1', 'ep_SHIP_LON_1']

# prediction targets: main engine fuel flow meters
target_vars = ['ep_ME_FLW_SUP_1', 'ep_ME_FLW_SUP_2']

# consolidated feature set for downstream validation
required_features = navigation_vars + power_vars + weather_vars + target_vars + position_vars

# vessel dataframe raw repository
dfs_ships_raw = {}
dfs_ships = {} # will be populated after temporal filtering

# port reference table ingestion
df_port = pd.read_csv(file_path2, sep=',', low_memory=False)
print("port data loaded")

for vessel_name, file_path in DataFrames.items():
    print(f"\nloading: {vessel_name}")

    # raw ingestion with encoding tolerance
    df_raw = pd.read_csv(file_path, sep=',', low_memory=False, encoding='latin1')
    df_raw.columns = df_raw.columns.str.strip()

    # timestamp parsing with mixed format support
    df_raw['ts'] = pd.to_datetime(df_raw['ts'], dayfirst=True, format='mixed')
    
    # store in raw dictionary
    dfs_ships_raw[vessel_name] = df_raw
    print(f"   loaded: {len(df_raw):,} records")



port data loaded

loading: ECO ADRIATICA
   loaded: 521,029 records

loading: ECO LIVORNO
   loaded: 431,033 records


### Temporale Configuration: Time Selection and Missing Timestamps Verification

In [4]:
DATE_START = pd.Timestamp("2026-02-23 00:00:00")
DATE_END = pd.Timestamp("2026-03-02 23:59:59")

for vessel_name in dfs_ships_raw.keys():
    df_filtered = dfs_ships_raw[vessel_name][
        (dfs_ships_raw[vessel_name]["ts"] >= DATE_START)
        & (dfs_ships_raw[vessel_name]["ts"] <= DATE_END)
    ].copy()
    dfs_ships[vessel_name] = df_filtered

    full_range = pd.date_range(
        start=df_filtered["ts"].min(), end=df_filtered["ts"].max(), freq="2min"
    )
    missing_count = len(full_range) - df_filtered["ts"].nunique()
    print(f"{vessel_name}: {missing_count} missing timestamps")


ECO ADRIATICA: 28 missing timestamps
ECO LIVORNO: 25 missing timestamps


## DATA CLEANING


### Raw Telemetry Negative Value Diagnostics

In [5]:
required_vars = navigation_vars + power_vars + weather_vars + target_vars

for name, df in dfs_ships.items():
    print(f"Vessel: {name} - Negative values per variable:")
    selected_cols = [
        col
        for col in required_vars
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col])
    ]
    for col in selected_cols:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            print(f"  - {col}: {neg_count} negative values")
    print("-" * 50)


Vessel: ECO ADRIATICA - Negative values per variable:
  - ep_SHIP_STW_1: 461 negative values
  - ep_SHG_POW_1: 1 negative values
  - ep_SHG_POW_2: 1 negative values
  - ep_ME_FLW_SUP_1: 976 negative values
  - ep_ME_FLW_SUP_2: 1720 negative values
--------------------------------------------------
Vessel: ECO LIVORNO - Negative values per variable:
  - ep_SHIP_STW_1: 1574 negative values
  - ep_SHG_POW_1: 1 negative values
  - ep_SHG_POW_2: 104 negative values
  - ep_ME_FLW_SUP_1: 2286 negative values
  - ep_ME_FLW_SUP_2: 1488 negative values
--------------------------------------------------


### Impossible Negative Sensor Values to NaN Conversion

In [6]:
for name in dfs_ships:
    df = dfs_ships[name]
    for var in required_vars:
        if (
            var in df.columns
            and var not in ["ep_WH_AIRT_1"]
        ):
            if pd.api.types.is_numeric_dtype(df[var]):
                df.loc[df[var] < 0, var] = np.nan


### Geographic Mapping and Voyage Identification

In [7]:
# Constants for coordinate columns
COL_LAT = 'ep_SHIP_LAT_1'  
COL_LON = 'ep_SHIP_LON_1'  

# Results container for navigation data
dfs_nav_ships = {}

# Define bounding boxes for geographical port identification
port_bboxes = {}
for index, row in df_port.iterrows():
    port_name = str(row['Port.Name']).strip()
    # Calculate boundaries for the port area
    lat_min, lat_max = min(row['Lat1'], row['Lat2']), max(row['Lat1'], row['Lat2'])
    lon_min, lon_max = min(row['Lon1'], row['Lon2']), max(row['Lon1'], row['Lon2'])
    port_bboxes[port_name] = {'lat_min': lat_min, 'lat_max': lat_max, 'lon_min': lon_min, 'lon_max': lon_max}

def identify_port_area_vectorized(df, lat_col, lon_col, bboxes):
    """Vectorized port area identification for high performance"""
    positions = pd.Series("Navigation", index=df.index)
    unknown_mask = df[lat_col].isna() | df[lon_col].isna()
    positions[unknown_mask] = "Unknown"
    for name, bbox in bboxes.items():
        mask = (
            (df[lat_col] >= bbox['lat_min']) & (df[lat_col] <= bbox['lat_max']) &
            (df[lon_col] >= bbox['lon_min']) & (df[lon_col] <= bbox['lon_max'])
        )
        positions[mask] = name
    return positions

# Process each vessel to identify navigation segments and routes
for vessel_name, df_raw in dfs_ships.items():
    vessel_start_time = time.time()
    print(f"\nAnalyzing voyages for: {vessel_name}")

    # Map geographical position to port or open sea (vectorized classification)
    df_raw['Current_Position'] = identify_port_area_vectorized(df_raw, COL_LAT, COL_LON, port_bboxes)

    # Reconstruct route history
    df_raw['Port_Point'] = df_raw['Current_Position'].replace(["Navigation", "Unknown"], pd.NA)
    df_raw['Departure_Port'] = df_raw['Port_Point'].ffill()
    df_raw['Arrival_Port'] = df_raw['Port_Point'].bfill()
    df_raw['Route'] = df_raw['Departure_Port'].astype(str) + " to " + df_raw['Arrival_Port'].astype(str)

    # Define engine operational states and motion thresholds
    active_engines = (
        (df_raw['ep_SHA_POW_1'] > 100) | (df_raw['ep_SHA_POW_2'] > 100) | 
        (df_raw['ep_SHA_SPEED_1'] > 10) | (df_raw['ep_SHA_SPEED_2'] > 10)
    )
    vessel_moving = (df_raw['ep_SHIP_SOG_1'] > 4) | active_engines

    # Filter for valid navigation segments
    df_nav_filtered = df_raw[
        (df_raw['Current_Position'] == "Navigation") & 
        vessel_moving &
        (df_raw['Departure_Port'].notna()) & 
        (df_raw['Departure_Port'] != df_raw['Arrival_Port'])
    ].copy()

    if df_nav_filtered.empty:
        print(f"No valid navigation segments for {vessel_name}")
        continue

    # Assign unique Voyage IDs based on temporal gaps exceeding 30 minutes
    df_nav_filtered = df_nav_filtered.sort_values('ts').reset_index(drop=True)
    df_nav_filtered['Voyage_ID'] = (df_nav_filtered['ts'].diff() > pd.Timedelta(minutes=30)).cumsum()
    
    # Prune nan-to-nan segments
    df_nav = df_nav_filtered[df_nav_filtered['Route'] != "nan to nan"].reset_index(drop=True)
    dfs_nav_ships[vessel_name] = df_nav
    print(f"Voyage analysis completed in {(time.time() - vessel_start_time):.2f} seconds")



Analyzing voyages for: ECO ADRIATICA
Voyage analysis completed in 0.68 seconds

Analyzing voyages for: ECO LIVORNO
Voyage analysis completed in 0.53 seconds


### Consecutive voyages merge and raw data restoration

In [8]:
print("Executing telemetry restoration and voyage merging")

for vessel_name in list(dfs_nav_ships.keys()):
    df_raw = dfs_ships[vessel_name].copy()
    df_nav = dfs_nav_ships[vessel_name].copy()
    
    # extract chronological list of active voyage ids
    voyage_ids = df_nav.groupby('Voyage_ID')['ts'].min().sort_values().index.tolist()
    
    merged_any = True
    while merged_any:
        merged_any = False
        i = 0
        while i < len(voyage_ids) - 1:
            v_prev = voyage_ids[i]
            v_curr = voyage_ids[i+1]
            
            df_prev = df_nav[df_nav['Voyage_ID'] == v_prev]
            df_curr = df_nav[df_nav['Voyage_ID'] == v_curr]
            
            route_prev = df_prev['Route'].iloc[0]
            route_curr = df_curr['Route'].iloc[0]
            
            # check for consecutive identical routes
            if route_prev == route_curr:
                ts_end_prev = df_prev['ts'].max()
                ts_start_curr = df_curr['ts'].min()
                gap_duration = ts_start_curr - ts_end_prev
                
                # slice the raw dataset to extract intermediate gap rows
                df_gap = df_raw[(df_raw['ts'] > ts_end_prev) & (df_raw['ts'] < ts_start_curr)].copy()
                
                # check for port visits in the gap to ensure it was a navigation slowdown
                visited_ports = df_gap[df_gap['Current_Position'] != 'Navigation']['Current_Position'].dropna().unique()
                
                # check for telemetry blackouts inside the gap
                if len(df_gap) > 0:
                    max_gap_step = df_gap['ts'].diff().max()
                else:
                    max_gap_step = gap_duration
                
                # condition gap under 12h no port visits and telemetry is continuous
                if (gap_duration < pd.Timedelta(hours=12)) and (len(visited_ports) == 0) and (max_gap_step < pd.Timedelta(hours=2)):
                                        
                    # fill voyage id and route for the restored gap rows
                    df_gap['Voyage_ID'] = v_prev
                    df_gap['Route'] = route_prev
                    
                    # merge current voyage into the previous one
                    df_nav.loc[df_nav['Voyage_ID'] == v_curr, 'Voyage_ID'] = v_prev
                    
                    # append restored rows back to the active dataset
                    df_nav = pd.concat([df_nav, df_gap], ignore_index=True)
                    df_nav = df_nav.sort_values(by='ts').reset_index(drop=True)
                    
                    voyage_ids = df_nav.groupby('Voyage_ID')['ts'].min().sort_values().index.tolist()
                    merged_any = True
                    break
            i += 1
            
    # re index voyage ids continuously starting from 1
    unique_voyages = df_nav.groupby('Voyage_ID')['ts'].min().sort_values().index.tolist()
    mapping_ids = {old_id: new_id + 1 for new_id, old_id in enumerate(unique_voyages)}
    df_nav['Voyage_ID'] = df_nav['Voyage_ID'].map(mapping_ids)
    
    dfs_nav_ships[vessel_name] = df_nav
    print(f"\nMerged routes completed for {vessel_name}. Continuous voyages: {df_nav['Voyage_ID'].nunique()}")


Executing telemetry restoration and voyage merging

Merged routes completed for ECO ADRIATICA. Continuous voyages: 7

Merged routes completed for ECO LIVORNO. Continuous voyages: 8


### Voyage Clustering and Operational Profiling

In [9]:
vessel = list(dfs_nav_ships.keys())[0]
df_nav = dfs_nav_ships[vessel]
primo_viaggio = df_nav[df_nav['Voyage_ID'] == df_nav['Voyage_ID'].iloc[0]]
print(primo_viaggio[['ts', 'miglia']].head(50).to_string())

                    ts    miglia
0  2026-02-23 20:31:39  0.450812
1  2026-02-23 20:33:40  0.508570
2  2026-02-23 20:35:40  0.513360
3  2026-02-23 20:37:40  0.524734
4  2026-02-23 20:39:40  0.521412
5  2026-02-23 20:41:40  0.529417
6  2026-02-23 20:43:40  0.541905
7  2026-02-23 20:45:40  0.553575
8  2026-02-23 20:47:41  0.547683
9  2026-02-23 20:49:41  0.558823
10 2026-02-23 20:51:41  0.551000
11 2026-02-23 20:53:41  0.559900
12 2026-02-23 20:55:41  0.555579
13 2026-02-23 20:57:41  0.555988
14 2026-02-23 20:59:41  0.556499
15 2026-02-23 21:01:41  0.557765
16 2026-02-23 21:03:41  0.559147
17 2026-02-23 21:05:41  0.564173
18 2026-02-23 21:07:41  0.555479
19 2026-02-23 21:09:42  0.563117
20 2026-02-23 21:11:42  0.550985
21 2026-02-23 21:13:42  0.558585
22 2026-02-23 21:15:42  0.553494
23 2026-02-23 21:17:42  0.550909
24 2026-02-23 21:19:42  0.552288
25 2026-02-23 21:21:42  0.551208
26 2026-02-23 21:23:42  0.551413
27 2026-02-23 21:25:42  0.552011
28 2026-02-23 21:27:42  0.553364
29 2026-02

In [10]:
# initialize data collection for voyage metrics
voyage_metrics = []

for vessel_name, df_nav in dfs_nav_ships.items():
    for v_id, group in df_nav.groupby('Voyage_ID'):
        # calculate trip duration in decimal hours
        duration = (group['ts'].max() - group['ts'].min()).total_seconds() / 3600.0
        
        # distance calculation logic
        # the miglia variable represents incremental distance covered in each two minute sampling interval
        # we sum these discrete increments across the trip to obtain total nautical miles
        distance = group['miglia'].sum()
            
        voyage_metrics.append({
            'Vessel': vessel_name, 'Voyage_ID': v_id, 'Route': group['Route'].iloc[0],
            'Duration_Hours': duration, 'Nautical_Miles': distance
        })

df_voyage_summary = pd.DataFrame(voyage_metrics)

df_voyage_summary = df_voyage_summary[
    (df_voyage_summary['Nautical_Miles'] / df_voyage_summary['Duration_Hours'] > 5) & 
    (df_voyage_summary['Nautical_Miles'] / df_voyage_summary['Duration_Hours'] < 25)
].copy()

# feature scaling for k means input
X = df_voyage_summary[['Duration_Hours', 'Nautical_Miles']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# optimization analysis via elbow method and silhouette score
wcss, sil_scores = [], []
k_range = range(2, 8)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

# automatic selection of optimal k based on maximum silhouette score
optimal_k = k_range[np.argmax(sil_scores)]

# plotting optimization metrics
fig_opt = make_subplots(specs=[[{"secondary_y": True}]])
fig_opt.add_trace(go.Scatter(x=list(k_range), y=wcss, name="WCSS (Elbow)"), secondary_y=False)
fig_opt.add_trace(go.Scatter(x=list(k_range), y=sil_scores, name="Silhouette Score"), secondary_y=True)
fig_opt.update_layout(title="K-Means Optimization: Elbow vs Silhouette", template="plotly_dark", xaxis_title="k")
fig_opt.show()

# final k means application
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_voyage_summary['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# standardize cluster labels based on average duration
avg_dur = df_voyage_summary.groupby('Cluster_ID')['Duration_Hours'].mean().sort_values()
mapping = {old: f"{new+1}" for new, (old, val) in enumerate(avg_dur.items())}
df_voyage_summary['Cluster_Number'] = df_voyage_summary['Cluster_ID'].map(mapping)

# faceted scatter plot for result validation
fig_cluster = px.scatter(
    df_voyage_summary, x='Duration_Hours', y='Nautical_Miles', color='Cluster_Number',
    facet_col='Vessel', hover_data=['Route', 'Voyage_ID'],
    title="Voyage Distribution Analysis: Duration vs Nautical Miles",
    labels={'Cluster_Number': 'Cluster'}, template="plotly_dark"
)
fig_cluster.update_layout(title_x=0.5)
fig_cluster.show()

# final dictionary for report integration
cluster_dictionary = df_voyage_summary.set_index(['Vessel', 'Voyage_ID'])['Cluster_Number'].to_dict()


c:\Users\siani\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\siani\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\siani\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\siani\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Window

c:\Users\siani\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.



### Data refinement and missing value management

In [15]:
def interpolate_series(series):
    # Check for missing values (NaN) in the telemetry series
    is_missing_value_mask = series.isna()

    # Identify contiguous blocks of missing and present values
    contiguous_missing_group_ids = (
        is_missing_value_mask != is_missing_value_mask.shift()
    ).cumsum()

    # Calculate the size of each contiguous missing value gap
    consecutive_missing_counts = is_missing_value_mask.groupby(
        contiguous_missing_group_ids
    ).transform("sum")

    # Flag gaps containing 10 or more consecutive missing values
    large_gap_missing_mask = is_missing_value_mask & (
        consecutive_missing_counts >= 10
    )

    # Copy series to avoid modifying the original data in-place
    temporary_series_with_filled_endpoints = series.copy()

    # Temporarily fill large gaps to act as boundary endpoints for interpolation
    temporary_series_with_filled_endpoints[large_gap_missing_mask] = (
        temporary_series_with_filled_endpoints.ffill().bfill()
    )

    # Perform linear interpolation only on small gaps (size < 10)
    linearly_interpolated_series = (
        temporary_series_with_filled_endpoints.interpolate(
            method="linear", limit_area="inside"
        )
    )

    # Restore large gaps back to NaN to ensure they are dropped later
    linearly_interpolated_series[large_gap_missing_mask] = np.nan

    return linearly_interpolated_series


# Iterative data cleaning and NA management for each vessel
for vessel_name, df_raw in dfs_nav_ships.items():
    print(f"\nStarting data refinement for: {vessel_name}")

    # Set base columns and extract available active sensors
    base_cols = ["ts", "Voyage_ID", "Route"]
    sensor_vars = [c for c in required_features if c in df_raw.columns]
    df_nav = df_raw[base_cols + sensor_vars].copy()

    # Drop columns that are completely null across the dataset
    null_cols = (
        df_nav[sensor_vars]
        .columns[df_nav[sensor_vars].isnull().all()]
        .tolist()
    )
    if null_cols:
        print(f"Dropping fully null columns: {null_cols}")
        df_nav.drop(columns=null_cols, inplace=True)
        sensor_vars = [v for v in sensor_vars if v not in null_cols]

    # Clean non-numeric text components and standardize decimals
    for var in sensor_vars:
        if df_nav[var].dtype == "object":
            text_values = (
                df_nav[var]
                .astype(str)
                .str.replace(r"[0-9.,-]", "", regex=True)
                .unique()
            )
            identified_text = [
                t.strip()
                for t in text_values
                if t.strip() not in ["", "nan", "None"]
            ]
            if identified_text:
                print(
                    f"   -> Variable {var}: Found text artifacts {identified_text}. Extracting numeric components..."
                )
                df_nav[var] = (
                    df_nav[var]
                    .astype(str)
                    .str.replace(r"[^0-9.,-]", "", regex=True)
                )
            df_nav[var] = df_nav[var].astype(str).str.replace(",", ".", regex=False)
        df_nav[var] = pd.to_numeric(df_nav[var], errors="coerce")

    # Drop inactive columns, constant columns, or columns filled with zeros
    invalid_cols = []
    for var in sensor_vars:
        if df_nav[var].isna().all():
            invalid_cols.append(var)
        elif (df_nav[var].dropna() == 0).all():
            invalid_cols.append(var)
        elif df_nav[var].dropna().nunique() <= 1:
            invalid_cols.append(var)

    if invalid_cols:
        print(f"   -> Dropping inactive or constant columns: {invalid_cols}")
        df_nav.drop(columns=invalid_cols, inplace=True)
        sensor_vars = [v for v in sensor_vars if v not in invalid_cols]

    print("Executing conditional linear interpolation (< 10 consecutive NaNs)...")

    # Apply the conditional interpolation per voyage segment to prevent leakage
    for var in sensor_vars:
        df_nav[var] = df_nav.groupby("Voyage_ID")[var].transform(
            interpolate_series
        )

    # Remove rows with residual NaNs belonging to un-interpolated large gaps
    rows_before = len(df_nav)
    df_nav = df_nav.dropna(subset=sensor_vars).reset_index(drop=True)
    rows_after = len(df_nav)
    print(
        f"\nNA management completed. Rows removed: {rows_before - rows_after}"
    )

    # Verify that all missing values have been successfully handled
    final_na_count = df_nav[sensor_vars].isna().sum().sum()
    print(f"Residual NAs for {vessel_name}: {final_na_count}")

    # Save the cleaned dataset back to the dictionary
    dfs_nav_ships[vessel_name] = df_nav



Starting data refinement for: ECO ADRIATICA
Executing conditional linear interpolation (< 10 consecutive NaNs)...

NA management completed. Rows removed: 0
Residual NAs for ECO ADRIATICA: 0

Starting data refinement for: ECO LIVORNO
Executing conditional linear interpolation (< 10 consecutive NaNs)...

NA management completed. Rows removed: 0
Residual NAs for ECO LIVORNO: 0


### Multi-Stage Anomaly Detection and Data Refinement


In [16]:
dfs_nav_clean_ships = {}
anomaly_summary = []

for vessel_name, df_source in dfs_nav_ships.items():
    print(f"processing vessel: {vessel_name}")
    df_vessel = df_source.copy()

    fuel_cols            = [v for v in target_vars      if 'FLW' in v and v in df_vessel.columns]
    nav_vars             = [v for v in navigation_vars  if v in df_vessel.columns]
    weather_vars_active  = [v for v in weather_vars     if v in df_vessel.columns]
    sensor_vars          = nav_vars + weather_vars_active
    
    power_vars_active    = [v for v in power_vars       if v in df_vessel.columns]
    all_features_to_check = sensor_vars + fuel_cols + power_vars_active

    print(f"calculating global iqr ranges for {vessel_name.lower()}")
    global_ranges = {}
    for col in all_features_to_check:
        series_clean = df_vessel[col].dropna()
        if len(series_clean) > 0:
            q1 = series_clean.quantile(0.25)
            q3 = series_clean.quantile(0.75)
            iqr = q3 - q1
            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr
            is_positive_only = any(x in col for x in ['DRAFT', 'SOG', 'STW', 'FLW', 'POW', 'SHA'])
            if is_positive_only and lower_bound < 0:
                lower_bound = 0.0
            global_ranges[col] = (lower_bound, upper_bound)
            print(f"   {col}: [{lower_bound:.4f}, {upper_bound:.4f}]")
        else:
            global_ranges[col] = (-np.inf, np.inf)

    for col in all_features_to_check:
        lower_bound, upper_bound = global_ranges[col]
        physical_mask = (df_vessel[col] >= lower_bound) & (df_vessel[col] <= upper_bound)
        df_vessel.loc[~physical_mask, col] = np.nan

    processed_voyages = []
    for v_id, df_voyage in df_vessel.groupby('Voyage_ID'):
        df_voy = df_voyage.copy()
        
        for col in all_features_to_check:
            series_clean = df_voy[col]
            
            rolling_median = series_clean.rolling(window=9, center=True, min_periods=1).median()
            q1 = series_clean.rolling(window=9, center=True, min_periods=1).quantile(0.25)
            q3 = series_clean.rolling(window=9, center=True, min_periods=1).quantile(0.75)
            rolling_iqr = q3 - q1
            
            active_median = series_clean.median()
            min_iqr = 0.05 * active_median if (not pd.isna(active_median) and active_median > 0) else 0.05
            rolling_iqr = np.maximum(rolling_iqr, min_iqr)
            
            lower_bound = rolling_median - 1.5 * rolling_iqr
            upper_bound = rolling_median + 1.5 * rolling_iqr
            
            outlier_mask = (series_clean < lower_bound) | (series_clean > upper_bound)
            df_voy.loc[outlier_mask, col] = np.nan

        IF_MIN_ROWS = 20
        if len(df_voy) >= IF_MIN_ROWS:
            X_if = df_voy[all_features_to_check].ffill().fillna(0)
            iso_pre = IsolationForest(n_estimators=100, contamination=0.03, random_state=42)
            if_flags = iso_pre.fit_predict(X_if) == -1
            df_voy.loc[if_flags, all_features_to_check] = np.nan

        processed_voyages.append(df_voy)

    df_vessel = pd.concat(processed_voyages).sort_values('ts')
    df_vessel['to_remove_power'] = False

    if power_vars_active:
        sha_cols = [c for c in ['ep_SHA_POW_1', 'ep_SHA_POW_2'] if c in df_vessel.columns]
        shg_cols = [c for c in ['ep_SHG_POW_1', 'ep_SHG_POW_2'] if c in df_vessel.columns]
        
        df_vessel['SHA_POW_TOT'] = df_vessel[sha_cols].sum(axis=1)
        df_vessel['SHG_POW_TOT'] = df_vessel[shg_cols].sum(axis=1)
        
        scaler = StandardScaler()
        pow_scaled = scaler.fit_transform(df_vessel[['SHA_POW_TOT', 'SHG_POW_TOT']].ffill().fillna(0))
        
        k_range = range(2, 6)
        scores = [
            silhouette_score(
                pow_scaled, 
                KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(pow_scaled), 
                sample_size=5000
            ) for k in k_range
        ]
        optimal_k = k_range[np.argmax(scores)]
        
        kmeans = KMeans(n_clusters=optimal_k, n_init=10, random_state=42).fit(pow_scaled)
        distances = kmeans.transform(pow_scaled)
        
        df_vessel['state_label'] = kmeans.labels_
        df_vessel['dist_to_centroid'] = [distances[i, label] for i, label in enumerate(kmeans.labels_)]
        df_vessel['to_remove_power'] = df_vessel.groupby('state_label')['dist_to_centroid'].transform(
            lambda x: x > x.quantile(0.98)
        )
        
        df_vessel.loc[df_vessel['to_remove_power'], all_features_to_check] = np.nan

    processed_clean_voyages = []
    for v_id, df_voy in df_vessel.groupby('Voyage_ID'):
        df_voy_clean = df_voy.copy()
        
        for col in all_features_to_check:
            df_voy_clean[col] = df_voy_clean[col].interpolate(method='linear', limit=10, limit_area='inside')
            
        df_voy_clean = df_voy_clean.dropna(subset=all_features_to_check)
        processed_clean_voyages.append(df_voy_clean)

    if processed_clean_voyages:
        df_clean = pd.concat(processed_clean_voyages).sort_values('ts').reset_index(drop=True)
        
        drop_cols = [c for c in df_clean.columns if any(x in c for x in ['to_remove', 'dist_to', 'state_label'])]
        df_clean.drop(columns=drop_cols, inplace=True, errors='ignore')
        
        dfs_nav_clean_ships[vessel_name] = df_clean
        
        original_rows = len(df_source)
        cleaned_rows = len(df_clean)
        removed_rows = original_rows - cleaned_rows
        retention_rate = (cleaned_rows / original_rows) * 100
        
        anomaly_summary.append({
            'vessel': vessel_name,
            'original_rows': original_rows,
            'cleaned_rows': cleaned_rows,
            'removed_anomalies': removed_rows,
            'retention_rate': f"{retention_rate:.1f}%"
        })

print("anomaly detection execution completed")
print(pd.DataFrame(anomaly_summary).to_string(index=False))


processing vessel: ECO ADRIATICA
calculating global iqr ranges for eco adriatica
   ep_SHIP_SOG_1: [8.4248, 27.7061]
   ep_SHIP_STW_1: [9.0958, 27.1958]
   ep_SHIP_DRAFTAFT_1: [5.1432, 7.1794]
   ep_SHIP_DRAFTFOR_1: [5.2925, 7.8783]
   ep_SHIP_HEAD_1: [-200.4000, 467.6000]
   miglia: [0.2826, 0.9225]
   ep_WH_WAVEH_1: [-0.0350, 0.6450]
   ep_WH_WAVEP_1: [1.3000, 5.3000]
   ep_WH_SWELLH_1: [-0.0298, 0.4603]
   ep_WH_SWELLP_1: [2.4000, 4.8000]
   ep_WH_SPEED_1: [-0.1595, 0.4414]
   ep_SHIP_SEAF_1: [3.0000, 3.0000]
   ep_SHIP_SEADIR_1: [-242.5000, 497.5000]
   ep_WH_DIR_1: [23.0000, 431.0000]
   ep_WH_AIRT_1: [11.3000, 15.3000]
   ep_ME_FLW_SUP_1: [0.0000, 2.6510]
   ep_ME_FLW_SUP_2: [0.0000, 2.6275]
   ep_SHA_POW_1: [0.0000, 13231.6927]
   ep_SHA_POW_2: [0.0000, 13423.3333]
   ep_SHG_POW_1: [489.2240, 1053.2656]
   ep_SHG_POW_2: [481.2240, 1080.0990]
processing vessel: ECO LIVORNO
calculating global iqr ranges for eco livorno
   ep_SHIP_SOG_1: [4.1095, 29.5537]
   ep_SHIP_STW_1: [4.6908,

### Main Engine Flow Meter Outlier Filtering & Signal Regularization

In [17]:
def clean_drop_outliers(df, col):
    y = df[col].values.copy()
    n = len(y)
    for i in range(1, n - 1):
        prev_val = y[i-1]
        curr_val = y[i]
        next_val = y[i+1]
        if prev_val > 0.8:
            if curr_val <= 0.5 * prev_val:
                if next_val >= 0.7 * prev_val:
                    y[i] = (prev_val + next_val) / 2.0
    return y

for name in dfs_nav_clean_ships:
    df = dfs_nav_clean_ships[name]
    df['ep_ME_FLW_SUP_1'] = clean_drop_outliers(df, 'ep_ME_FLW_SUP_1')
    df['ep_ME_FLW_SUP_2'] = clean_drop_outliers(df, 'ep_ME_FLW_SUP_2')


### Cleaned Telemetry Multi-Variable Visual Inspection & Statistical Distributions

In [19]:
start_date = "2026-02-23 00:00:00"
end_date = "2026-03-02 00:00:00"

for vessel_name in dfs_nav_clean_ships.keys():
    df_clean = dfs_nav_clean_ships[vessel_name].copy().sort_values('ts')
    df_clean['ts'] = pd.to_datetime(df_clean['ts'])
    
    # calculate total powers on the fully cleaned and interpolated data
    sha_cols = [c for c in ['ep_SHA_POW_1', 'ep_SHA_POW_2'] if c in df_clean.columns]
    shg_cols = [c for c in ['ep_SHG_POW_1', 'ep_SHG_POW_2'] if c in df_clean.columns]
    
    if sha_cols:
        df_clean['SHA_POW_TOT'] = df_clean[sha_cols].sum(axis=1)
    if shg_cols:
        df_clean['SHG_POW_TOT'] = df_clean[shg_cols].sum(axis=1)
    
    df_slice = df_clean[(df_clean['ts'] >= start_date) & (df_clean['ts'] <= end_date)].copy()
    
    if df_slice.empty:
        print(f"no samples found for {vessel_name} in range {start_date} to {end_date}")
        continue
        
    print(f"generating comprehensive visual and statistical diagnostics for {vessel_name}")
    
    exclude_cols = ['ts', 'Voyage_ID', 'Route', 'Departure_Port', 'Arrival_Port', 'Current_Position', 'Cluster_Number']
    
    # hide individual generator columns to focus on the totals
    individual_power_cols = ['ep_SHA_POW_1', 'ep_SHA_POW_2', 'ep_SHG_POW_1', 'ep_SHG_POW_2', 'ep_DG_POW_1', 'ep_DG_POW_2', 'ep_DG_POW_3', 'DG_POW_TOT']
    exclude_cols.extend([c for c in individual_power_cols if c in df_slice.columns])
    
    plot_cols = [c for c in df_slice.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df_slice[c])]
    
    n_plots = len(plot_cols)
    
    subplot_titles = []
    for col in plot_cols:
        subplot_titles.extend([f"{col.lower()} time series", f"{col.lower()} distribution"])
        
    fig = make_subplots(
        rows=n_plots, 
        cols=2, 
        column_widths=[0.75, 0.25], 
        shared_xaxes=False, 
        shared_yaxes=True,
        vertical_spacing=0.02, 
        horizontal_spacing=0.03, 
        subplot_titles=subplot_titles
    )
    
    for i, col in enumerate(plot_cols):
        r = i + 1
        
        fig.add_trace(
            go.Scattergl(
                x=df_slice['ts'], 
                y=df_slice[col],
                name=f"{col.lower()} ts",
                line=dict(width=1.1, color='#0284c7'),
                connectgaps=False,
                showlegend=False
            ),
            row=r, 
            col=1
        )
        
        fig.add_trace(
            go.Histogram(
                y=df_slice[col],
                name=f"{col.lower()} dist",
                marker_color='#0f172a',
                opacity=0.8,
                showlegend=False
            ),
            row=r, 
            col=2
        )
        
    fig.update_layout(
        template='plotly_white',
        paper_bgcolor='#ffffff',
        plot_bgcolor='#f8fafc',
        font=dict(family='Plus Jakarta Sans, sans-serif', color='#0f172a', size=11),
        margin=dict(l=60, r=30, t=60, b=60),
        height=max(600, n_plots * 160),
        hovermode='closest',
        title=dict(
            text=f"telemetry integrity and statistical distribution audit for {vessel_name.lower()}",
            font=dict(size=18, color='#0f172a', weight='bold')
        )
    )
    
    for r in range(1, n_plots + 1):
        fig.update_yaxes(gridcolor='#e2e8f0', zerolinecolor='#cbd5e1', row=r, col=1)
        fig.update_xaxes(gridcolor='#e2e8f0', zerolinecolor='#cbd5e1', row=r, col=1)
        fig.update_yaxes(gridcolor='#e2e8f0', zerolinecolor='#cbd5e1', row=r, col=2)
        fig.update_xaxes(gridcolor='#e2e8f0', zerolinecolor='#cbd5e1', row=r, col=2)
        
    fig.update_xaxes(title_text="acquisition timestamp", row=n_plots, col=1)
    fig.update_xaxes(title_text="frequency count", row=n_plots, col=2)
    
    fig.show()


generating comprehensive visual and statistical diagnostics for ECO ADRIATICA


generating comprehensive visual and statistical diagnostics for ECO LIVORNO


### Temporal Resampling and Final Dataset Export

In [20]:
start_time_aggregation = time.time()

dfs_nav_agg_ships = {}

output_folder = r"c:\users\siani\desktop\df_week"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

for vessel_name, df_clean_orig in dfs_nav_clean_ships.items():
    print(f"resampling and aggregating data for {vessel_name}")
 
    df_vessel = df_clean_orig.copy().reset_index(drop=True)
    df_vessel['ts'] = pd.to_datetime(df_vessel['ts'])

    sha_cols = [c for c in ['ep_SHA_POW_1', 'ep_SHA_POW_2'] if c in df_vessel.columns]
    shg_cols = [c for c in ['ep_SHG_POW_1', 'ep_SHG_POW_2'] if c in df_vessel.columns]
    
    if sha_cols:
        df_vessel['SHA_POW_TOT'] = df_vessel[sha_cols].sum(axis=1)
    if shg_cols:
        df_vessel['SHG_POW_TOT'] = df_vessel[shg_cols].sum(axis=1)

    agg_rules = {}
    for col in df_vessel.columns:
        if col in ['Voyage_ID', 'ts']: 
            continue
       
        elif col in ['Route', 'Current_Position', 'Departure_Port', 'Arrival_Port', 'Cluster_Number']:
            agg_rules[col] = 'first'
      
        elif 'State' in col or 'Cluster' in col:
            agg_rules[col] = lambda x: x.mode()[0] if not x.mode().empty else np.nan
       
        else:
            agg_rules[col] = 'median'

    df_agg = (
        df_vessel
        .set_index('ts')
        .groupby('Voyage_ID')
        .resample('15min') 
        .agg(agg_rules)
    )
   
    df_agg = df_agg.dropna(subset=['Route']).reset_index()
  
    df_final = df_agg.sort_values(by=['Voyage_ID', 'ts']).reset_index(drop=True)
    dfs_nav_agg_ships[vessel_name] = df_final

    safe_name = vessel_name.replace(" ", "_")
    file_path = os.path.join(output_folder, f"refined_15min_{safe_name}.csv")
    df_final.to_csv(file_path, index=False)
    
    print(f"successfully exported samples for {vessel_name}")

total_time = time.time() - start_time_aggregation
print(f"aggregation task completed in {total_time:.2f} seconds all files saved to {output_folder}")


resampling and aggregating data for ECO ADRIATICA
successfully exported samples for ECO ADRIATICA
resampling and aggregating data for ECO LIVORNO
successfully exported samples for ECO LIVORNO
aggregation task completed in 0.93 seconds all files saved to c:\users\siani\desktop\df_week
